# FoodIntent: classificação de intenções de suporte

Experimento reproduzível em português com TF-IDF, Logistic Regression e Multinomial Naive Bayes. Execute todas as células para obter métricas reais; não há resultados pré-preenchidos.

> O SID é um dataset sintético multilíngue. Os resultados não estimam desempenho em conversas reais de delivery.

## 1. Carregar e filtrar dados

Habilite Internet nas configurações do notebook Kaggle para baixar os arquivos públicos.

In [ ]:
import pandas as pd
from IPython.display import display

BASE_URL = "https://huggingface.co/datasets/luigicfilho/sid/resolve/main"
SPLITS = {"train": "train.csv", "validation": "val.csv", "test": "test.csv"}
LABEL_MAP = {
    "track_order": "order_status", "check_order_status": "order_status",
    "cancel_order": "cancellation",
    "report_payment_failure": "payment_issue", "dispute_charge": "payment_issue",
    "request_refund": "refund_request", "report_delivery_issue": "delivery_problem",
}
TARGET_NAMES = sorted(set(LABEL_MAP.values()))

def load_split(name):
    frame = pd.read_csv(f"{BASE_URL}/{SPLITS[name]}")
    frame = frame.loc[frame["language"].eq("pt")].copy()
    frame["intent"] = frame["label"].map(LABEL_MAP)
    frame = frame.dropna(subset=["intent"])
    frame["text"] = frame["text"].astype(str).str.strip()
    return frame.loc[frame["text"].ne("")][["text", "intent"]].reset_index(drop=True)

train_df = load_split("train")
validation_df = load_split("validation")
test_df = load_split("test")
print({"train": len(train_df), "validation": len(validation_df), "test": len(test_df)})
display(train_df.head())

## 2. Análise exploratória

Observe a distribuição das classes antes de avaliar o modelo. O mapeamento agrupa intenções relacionadas, mas não idênticas.

In [ ]:
display(train_df["intent"].value_counts().rename("examples").to_frame())
display(train_df.assign(message_length=train_df["text"].str.len()).groupby("intent")["message_length"].describe().round(1))

## 3. Comparar baselines

A macro F1 no conjunto de validação escolhe o modelo. O conjunto de teste fica reservado para uma única avaliação final.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

def make_pipeline(classifier):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000, strip_accents="unicode")), ("classifier", classifier)])

candidates = {
    "logistic_regression": make_pipeline(LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42)),
    "multinomial_nb": make_pipeline(MultinomialNB(alpha=0.5)),
}
scores = {}
for name, model in candidates.items():
    model.fit(train_df["text"], train_df["intent"])
    pred = model.predict(validation_df["text"])
    scores[name] = f1_score(validation_df["intent"], pred, labels=TARGET_NAMES, average="macro", zero_division=0)
    print(name, "validation macro F1:", round(scores[name], 4))

best_name = max(scores, key=scores.get)
best_model = candidates[best_name]
print("Selecionado:", best_name)

## 4. Avaliar no teste e analisar erros

Use os resultados efetivamente exibidos abaixo no artigo técnico.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

test_pred = best_model.predict(test_df["text"])
print("accuracy:", round(accuracy_score(test_df["intent"], test_pred), 4))
print("macro F1:", round(f1_score(test_df["intent"], test_pred, labels=TARGET_NAMES, average="macro", zero_division=0), 4))
print(classification_report(test_df["intent"], test_pred, labels=TARGET_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(test_df["intent"], test_pred, labels=TARGET_NAMES, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.tight_layout()
plt.show()

errors = test_df.loc[test_df["intent"].ne(test_pred)].copy()
errors["predicted_intent"] = test_pred[test_df["intent"].ne(test_pred)]
display(errors.head(20))

## 5. Limitações e fonte

O SID é sintético e multilíngue. `report_delivery_issue` é uma categoria ampla, não necessariamente um pedido sobre item incorreto. O rótulo `order_status` combina acompanhamento e consulta de pedido, que não implicam atraso. A avaliação mede apenas o benchmark filtrado. Um próximo passo é criar uma amostra pt-BR de delivery revisada por pessoas, com licença e anotação documentadas.

Fonte: [luigicfilho/sid](https://huggingface.co/datasets/luigicfilho/sid), licença MIT. Consulte o cartão do dataset antes de redistribuir dados.